# ZK-LLM-Turbo: Privacy-Preserving Split Inference Demo

This notebook demonstrates the core privacy guarantees of ZK-LLM-Turbo:
- **Privacy**: Server receives only encrypted data with a public key (cannot decrypt)
- **Correctness**: HE matrix multiplication matches plaintext computation
- **Architecture**: TinyLlama 1.1B with CKKS homomorphic encryption

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import tenseal as ts
import time
import torch

## 1. Privacy Guarantee: Server Cannot Decrypt

The client creates a CKKS context with a secret key, then serializes only the public key for the server. The server can perform computations but never sees plaintext.

In [ ]:
from client.encryption.ckks_context import create_ckks_context, serialize_public_context

# Client creates full context (has secret key)
client_ctx = create_ckks_context(config_path="../client/config/client_config.yaml")

# Serialize public-only context
public_bytes = serialize_public_context(client_ctx)
print(f"Public context size: {len(public_bytes) / 1024:.1f} KB")

# Server loads public context
server_ctx = ts.context_from(public_bytes)

# Client encrypts secret data
secret_data = [3.14, 2.71, 1.41, 1.73]
enc_vec = ts.ckks_vector(client_ctx, secret_data)

# Server receives ciphertext
server_vec = ts.ckks_vector_from(server_ctx, enc_vec.serialize())

# Server CANNOT decrypt - this will fail
try:
    server_vec.decrypt()
    print("ERROR: Server decrypted! This should not happen.")
except Exception as e:
    print(f"Server CANNOT decrypt (expected): {type(e).__name__}")

# Client CAN decrypt - this works
client_result = ts.ckks_vector_from(client_ctx, enc_vec.serialize())
decrypted = client_result.decrypt()[:4]
print(f"Client decrypts: {decrypted}")
print(f"Original data:   {secret_data}")

## 2. HE Matrix Multiplication

The server computes `Enc(x) @ W` homomorphically without ever seeing `x`.

In [ ]:
# Simulate a small projection: 64-dim input → 32-dim output
dim_in, dim_out = 64, 32
x = np.random.randn(dim_in).astype(np.float32) * 0.1
W = np.random.randn(dim_in, dim_out).astype(np.float32) * 0.1

# Plaintext result
expected = x @ W

# Client encrypts x
enc_x = ts.ckks_vector(client_ctx, x.tolist())

# Server computes enc_x @ W (homomorphically)
server_enc = ts.ckks_vector_from(server_ctx, enc_x.serialize())
t0 = time.perf_counter()
enc_result = server_enc.mm(W.tolist())
he_time = (time.perf_counter() - t0) * 1000

# Client decrypts
actual = np.array(ts.ckks_vector_from(client_ctx, enc_result.serialize()).decrypt()[:dim_out])

error = np.abs(actual - expected)
print(f"HE matmul ({dim_in}→{dim_out}): {he_time:.1f} ms")
print(f"Max error: {error.max():.6f}")
print(f"Mean error: {error.mean():.6f}")

## 3. Non-Linear Operations (Client-Side)

Non-linear operations (RMSNorm, SiLU, Softmax) run on the client where they can be computed efficiently in plaintext.

In [ ]:
from client.inference.nonlinear_ops import rms_norm, silu, softmax

# RMSNorm comparison
x_np = np.random.randn(2048).astype(np.float32)
w_np = np.random.randn(2048).astype(np.float32)

our_norm = rms_norm(x_np, w_np, eps=1e-5)

x_t = torch.tensor(x_np)
w_t = torch.tensor(w_np)
variance = x_t.pow(2).mean(-1, keepdim=True)
pt_norm = ((x_t * torch.rsqrt(variance + 1e-5)) * w_t).numpy()

print(f"RMSNorm max diff: {np.abs(our_norm - pt_norm).max():.2e}")

# SiLU comparison
x_np = np.random.randn(1000).astype(np.float32)
our_silu = silu(x_np)
pt_silu = torch.nn.functional.silu(torch.tensor(x_np)).numpy()
print(f"SiLU max diff: {np.abs(our_silu - pt_silu).max():.2e}")

# Softmax comparison
x_np = np.random.randn(32, 32).astype(np.float32)
our_sm = softmax(x_np)
pt_sm = torch.nn.functional.softmax(torch.tensor(x_np), dim=-1).numpy()
print(f"Softmax max diff: {np.abs(our_sm - pt_sm).max():.2e}")

## 4. Timing Breakdown

Per-token overhead breakdown for encrypted operations.

In [ ]:
hidden_dim = 2048
x = np.random.randn(hidden_dim).astype(np.float32) * 0.01

# Encryption time
t0 = time.perf_counter()
enc_x = ts.ckks_vector(client_ctx, x.tolist())
encrypt_ms = (time.perf_counter() - t0) * 1000

# Serialize time
t0 = time.perf_counter()
enc_bytes = enc_x.serialize()
serialize_ms = (time.perf_counter() - t0) * 1000

# Decrypt time
t0 = time.perf_counter()
dec = enc_x.decrypt()
decrypt_ms = (time.perf_counter() - t0) * 1000

print("=== Timing Breakdown (single token, 2048-dim) ===")
print(f"Encrypt:     {encrypt_ms:8.1f} ms")
print(f"Serialize:  {serialize_ms:8.1f} ms")
print(f"Decrypt:    {decrypt_ms:8.1f} ms")
print(f"Ciphertext size: {len(enc_bytes) / 1024:.1f} KB")

## Summary

| Feature | Details |
|---------|---------|
| **Privacy** | Server has public context only — cannot decrypt |
| **Correctness** | HE matmul matches plaintext within CKKS tolerance |
| **Non-linear ops** | Run client-side in plaintext for efficiency |
| **Protocol** | 4 round-trips per encrypted layer |
| **Model** | TinyLlama 1.1B (22 layers, 2048 hidden dim) |
| **Encryption** | CKKS via TenSEAL (8192 polynomial degree) |